# M&A Synergy Prediction: Final Visualizations
This notebook generates the four high-dimensional visualizations for the final dissertation:
1. **The Topological Alpha Ego-Network**
2. **The Multimodal SHAP Summary Plot**
3. **The H3 Volatility Funnel Scatterplot**
4. **The ROC-AUC Capability Gap**

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
import torch
import torch_geometric


# Ensure scripts path is available for imports
sys.path.insert(0, str(Path(os.getcwd()).parent / "scripts"))
try:
    from training_utils import load_and_prepare_data, get_feature_configs
except ImportError:
    sys.path.insert(0, str(Path(os.getcwd()) / "scripts"))
    from training_utils import load_and_prepare_data, get_feature_configs

# Global style
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(
    style="whitegrid",
    font_scale=1.05,
    rc={
        "axes.edgecolor": "#333333",
        "axes.linewidth": 0.8,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "grid.alpha": 0.2,
        "grid.linestyle": "-",
    },
)

# Define data path based on where notebook is run
data_path = "../data/processed/final_multimodal_dataset.csv"
if not os.path.exists(data_path):
    data_path = "data/processed/final_multimodal_dataset.csv"

# Create output directory for high-res images
fig_dir = "../docs/figures"
if not os.path.exists("../docs"):
    fig_dir = "docs/figures"
os.makedirs(fig_dir, exist_ok=True)

# Load main dataset once
subset, y_cont = load_and_prepare_data()
y_binary = (y_cont > 0).astype(int)
configs = get_feature_configs(subset)

X_m3 = subset[configs["M3"]["cols"]]
X_m3_imputed = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(X_m3),
    columns=X_m3.columns,
)


## 1. The "Topological Alpha" Ego-Network (Task 2.1)
Visualizing a complex supply chain network surrounding a specific acquisition to show the structural context.

In [ ]:
hetero_graph_path = "../data/interim/hetero_supply_chain_graph.pt"
hetero_meta_path = "../data/interim/hetero_graph_metadata.json"
if not os.path.exists(hetero_graph_path):
    hetero_graph_path = "data/interim/hetero_supply_chain_graph.pt"
    hetero_meta_path = "data/interim/hetero_graph_metadata.json"

graph_data = torch.load(hetero_graph_path, weights_only=False)
with open(hetero_meta_path, "r") as f:
    metadata = json.load(f)

homo_data = graph_data.to_homogeneous()
G_nx = torch_geometric.utils.to_networkx(homo_data, to_undirected=True)

degrees = dict(G_nx.degree())
hub = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[2][0]

neighbors_k = nx.single_source_shortest_path_length(G_nx, hub, cutoff=2)
sub_G = G_nx.subgraph(neighbors_k.keys()).copy()

plt.figure(figsize=(8, 8))
pos = nx.spring_layout(sub_G, seed=42, k=0.2)

betweenness = nx.betweenness_centrality(sub_G)
node_color = [betweenness[n] for n in sub_G.nodes()]

nodes = nx.draw_networkx_nodes(
    sub_G,
    pos,
    node_size=[40 + 600 * betweenness[n] for n in sub_G.nodes()],
    node_color=node_color,
    cmap="plasma",
    alpha=0.9,
)
nx.draw_networkx_edges(sub_G, pos, alpha=0.15, edge_color="#777777")

try:
    id_to_tkr = {v: k for k, v in metadata["ticker_to_id"].items()}
    hub_label = id_to_tkr.get(hub, str(hub))
except Exception:
    hub_label = str(hub)

nx.draw_networkx_nodes(
    sub_G,
    pos,
    nodelist=[hub],
    node_color="#ffdd00",
    edgecolors="black",
    node_size=900,
)
nx.draw_networkx_labels(
    sub_G,
    pos,
    labels={hub: hub_label},
    font_size=10,
    font_weight="semibold",
)

plt.title(
    f"Topological Alpha: Ego-Network for {hub_label}",
    fontsize=14,
    weight="semibold",
)
plt.colorbar(nodes, label="Betweenness Centrality", shrink=0.7)
plt.axis("off")
plt.tight_layout()
plt.savefig(
    f"{fig_dir}/topological_alpha_ego_network.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 2. The Multimodal SHAP Summary Plot (Task 2.2)
Proving that graph embeddings (graph_emb) and text embeddings (rf_pca) materially impact the predictions alongside financial ratios.

In [ ]:
model = XGBClassifier(eval_metric="auc", random_state=42, use_label_encoder=False)
model.fit(X_m3_imputed, y_binary)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_m3_imputed)

plt.figure(figsize=(7.5, 9))
shap.summary_plot(
    shap_values,
    X_m3_imputed,
    max_display=15,
    show=False,
    plot_size=None,
    cmap="Spectral_r",
    alpha=0.7,
)
plt.title(
    "Multimodal Feature Impact (SHAP Summary Plot)",
    fontsize=14,
    weight="semibold",
)
plt.tight_layout()
plt.savefig(
    f"{fig_dir}/shap_summary_plot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 3. The "H3 Volatility Funnel" Scatterplot (Task 2.3)
Scatterplot of Betweenness Centrality vs Absolute CAR with a seaborn KDE density overlay. Proving the Information Transparency Dampening Effect.

In [ ]:
hetero_graph_path = "../data/interim/hetero_supply_chain_graph.pt"
hetero_meta_path = "../data/interim/hetero_graph_metadata.json"
if not os.path.exists(hetero_graph_path):
    hetero_graph_path = "data/interim/hetero_supply_chain_graph.pt"
    hetero_meta_path = "data/interim/hetero_graph_metadata.json"

print("Loading graph for centrality computation...")
graph_data = torch.load(hetero_graph_path, weights_only=False)
with open(hetero_meta_path, "r") as f:
    metadata = json.load(f)

homo_data = graph_data.to_homogeneous()
G_nx = torch_geometric.utils.to_networkx(homo_data, to_undirected=True)

print("Computing Betweenness Centrality on full graph...")
centrality = nx.betweenness_centrality(G_nx)

ticker_to_id = metadata["ticker_to_id"]
deal_to_acq_ticker = metadata["deal_to_acq_ticker"]

records = []
for deal_id_str, acq_ticker in deal_to_acq_ticker.items():
    deal_id = int(deal_id_str)
    node_id = ticker_to_id.get(acq_ticker)
    if node_id is not None:
        records.append(
            {
                "deal_id": deal_id,
                "Acq_Betweenness": centrality.get(node_id, np.nan),
            }
        )

cent_df = pd.DataFrame(records).set_index("deal_id")

subset_with_cent = subset.join(cent_df, how="inner")

if not isinstance(y_cont, pd.Series):
    y_cont = pd.Series(y_cont, index=subset.index)

subset_with_cent["abs_car"] = np.abs(y_cont.loc[subset_with_cent.index])
plot_df = subset_with_cent[["Acq_Betweenness", "abs_car"]].dropna()

plt.figure(figsize=(10, 6))

sns.kdeplot(
    data=plot_df,
    x="Acq_Betweenness",
    y="abs_car",
    fill=True,
    cmap="Blues",
    alpha=0.25,
    levels=8,
    thresh=0.02,
    warn_singular=False,
)
sns.scatterplot(
    data=plot_df,
    x="Acq_Betweenness",
    y="abs_car",
    color="#1f3b7c",
    alpha=0.3,
    s=12,
    edgecolor="none",
)

sns.regplot(
    data=plot_df,
    x="Acq_Betweenness",
    y="abs_car",
    scatter=False,
    color="#d62728",
    line_kws={"linestyle": "--", "linewidth": 1.5},
)

plt.title(
    "H3 Volatility Funnel: Information Transparency Effect",
    fontsize=14,
    weight="semibold",
)
plt.xlabel("Acquirer Betweenness Centrality", fontsize=11)
plt.ylabel("Absolute CAR", fontsize=11)
plt.grid(alpha=0.2, linewidth=0.5)
plt.tight_layout()
plt.savefig(
    f"{fig_dir}/h3_volatility_funnel.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 4. The ROC-AUC Capability Gap (Task 2.4)
The ultimate proof of your dissertation adding value: plotting the AUC difference between Financial-only (M1) and Multimodal (M3) features.

In [ ]:
X_m1 = subset[configs["M1"]["cols"]]
X_m1_imputed = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(X_m1),
    columns=X_m1.columns,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def get_cv_predictions(X, y):
    y_probs = np.zeros(len(y))
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y.iloc[train_idx]

        model = XGBClassifier(
            eval_metric="auc", random_state=42, use_label_encoder=False
        )
        model.fit(X_train, y_train)

        preds = model.predict_proba(X_test)[:, 1]
        y_probs[test_idx] = preds
    return y_probs

y_binary_s = pd.Series(y_binary)
probs_m1 = get_cv_predictions(X_m1_imputed, y_binary_s)
probs_m3 = get_cv_predictions(X_m3_imputed, y_binary_s)

fpr_m1, tpr_m1, _ = roc_curve(y_binary, probs_m1)
fpr_m3, tpr_m3, _ = roc_curve(y_binary, probs_m3)

auc_m1 = auc(fpr_m1, tpr_m1)
auc_m3 = auc(fpr_m3, tpr_m3)

fpr_grid = np.linspace(0.0, 1.0, 1000)
tpr_m1_interp = np.interp(fpr_grid, fpr_m1, tpr_m1)
tpr_m3_interp = np.interp(fpr_grid, fpr_m3, tpr_m3)

plt.figure(figsize=(7.5, 7))
plt.plot(
    fpr_m3,
    tpr_m3,
    color="#ff7f0e",
    lw=2,
    label=f"M3 Multimodal (AUC = {auc_m3:.3f})",
)
plt.plot(
    fpr_m1,
    tpr_m1,
    color="#1f77b4",
    lw=2,
    label=f"M1 Financials Only (AUC = {auc_m1:.3f})",
)
plt.plot([0, 1], [0, 1], color="#999999", lw=1, linestyle="--")

plt.fill_between(
    fpr_grid,
    tpr_m1_interp,
    tpr_m3_interp,
    where=tpr_m3_interp >= tpr_m1_interp,
    color="#ffbb78",
    alpha=0.25,
    label=f'"Topological Alpha" Gap (ΔAUC = {auc_m3 - auc_m1:.3f})',
)

plt.xlabel("False Positive Rate", fontsize=11)
plt.ylabel("True Positive Rate", fontsize=11)
plt.title(
    "ROC-AUC Capability Gap: Multimodal vs Financial Baseline",
    fontsize=14,
    weight="semibold",
)
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.grid(alpha=0.2, linewidth=0.5)
plt.legend(frameon=False, fontsize=10, loc="lower right")
plt.tight_layout()
plt.savefig(
    f"{fig_dir}/roc_auc_gap.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()
